In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

/var/folders/lp/hgbg7xdj7ql_zq41nk0lt6yw0000gn/T/ipykernel_38704/3026728138.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## Step 1: Input Dataset Structure
- **Date:** The timestamp of the asset valuation. The data is logged quarterly (every 3 months), spanning from December 31, 2007, onwards.

- **Asset columns (General Electric, Chesapeake Energy, AT&T, Cisco Systems, Campbell Soup):** The numberical values represent the Adjusted Closing Price (in USD) of each company's stock at that specific quarter-end date.

In [ ]:
df = pd.read_csv("../../dataset/archive/stock_data_weak.csv", parse_dates=["Date"], index_col='Date')
prices = df.to_numpy()
prices

## Step 2: Theoretical Goal: Transforming Prices to Risk and Return

In optimization theory, raw stock prices cannot be directly injected into the Lagrangian function. Instead, we must compute two fundamental metrics to construct out objective funtion and constraits:

1. Expected Return Vector ($r$): Represents the average asset growth rate per period.

2. Covariance Matrix ($\Sigma$): Represents the asset variances (risk) and their co-movements.

### Calculate Percentage Periodic Returns
<img src="../asset/img/001_img_AA002FAI.png" width="800" height="200">

In [9]:
# Manually compute quarterly asset returns: R_t = (P_t - P_{t-1}) / P_{t-1}
returns = (prices[1:] - prices[:-1]) / prices[:-1]
returns

array([[-0.01213592,  0.16260163, -0.10470085, -0.11633028, -0.05199888],
       [-0.28064428,  0.45913462, -0.1315301 , -0.02533223, -0.02595105],
       [-0.09411765, -0.50142279, -0.13435115, -0.05198125,  0.15955192],
       [-0.3372434 , -0.53920096, -0.00564374, -0.27280899, -0.23002611],
       [-0.34702908,  0.1505867 , -0.09755232,  0.02163164, -0.08341811],
       [ 0.14617619,  0.11728045, -0.01179245,  0.14579552,  0.0972993 ],
       [ 0.4214527 ,  0.46044625,  0.08750994,  0.24340021,  0.11058665],
       [-0.09269162, -0.08472222,  0.03474762,  0.02335456,  0.03642987],
       [ 0.19253438, -0.107739  , -0.08448215,  0.10041494,  0.03251318],
       [-0.20538166, -0.08715986, -0.05984556, -0.18853695,  0.02099291],
       [ 0.13752592,  0.05542618,  0.18193018,  0.02462825,  0.0030564 ],
       [ 0.10267315,  0.14254192,  0.02015288, -0.08208617, -0.04432133],
       [ 0.10633609,  0.33565083,  0.04325613, -0.14426877, -0.03333333],
       [-0.07420319, -0.15615963,  0.0

In [11]:
T, n = returns.shape
T, n

(41, 5)

### Calculate the Expected Return Vector ($r$)
<img src="../asset/img/002_img_AA002FAI.png" width="800" height="200">

In [14]:
# Manually compute the Expected Return Vector (r)
r = np.sum(returns, axis = 0) / T
r

array([-0.01204791, -0.03060832, -0.00146363,  0.01847859,  0.00846942])

### Calculate the Covariance Matrix ($\Sigma$)
<img src="../asset/img/003_img_AA002FAI.png" width="800" height="200">

In [18]:
# Manually compute the Covariance Matrix (Sigma) using matrix multiplication
returns_centered = returns - r
Sigma = returns_centered.T @ returns_centered / (T - 1)
Sigma_inv = np.linalg.inv(Sigma)

## Step 3: Define the Lagrangian Minimizer Function
<img src="../asset/img/004_img_AA002FAI.png" width="600" height="400">

In [20]:
ones = np.ones(n)
R_target = 0.015

### The Lagrangian Function & Analytical Minimizer

By introducing the dual variables (Lagrange multipliers) $\lambda \ge 0$ for the inequality constraint and $\mu \in \mathbb{R}$ for the equality constraint, we remove the constraints by penalizing them directly in the objective function. This yields the Lagrangian Function $\mathcal{L}(x; \lambda, \mu)$:

$$\mathcal{L}(x; \lambda, \mu) = \frac{1}{2} x^T \Sigma x + \lambda (R_{\text{target}} - r^T x) + \mu (\mathbf{1}^T x - 1)$$

Deriving the Analytical Formula for $x^\star$

To evaluate the inner infimum ($\inf_x \mathcal{L}$), we take the first-order derivative (gradient) of the Lagrangian with respect to the primal variable vector $x$ and set it to zero:

$$\nabla_x \mathcal{L}(x; \lambda, \mu) = \Sigma x - \lambda r + \mu \mathbf{1} = 0$$

Isolating the variable vector $x$, we compute the direct analytical mapping:

$$\Sigma x = \lambda r - \mu \mathbf{1}$$

$$x^\star(\lambda, \mu) = \Sigma^{-1} (\lambda r - \mu \mathbf{1})$$

Practical Core Value: This exact matrix formulation eliminates the need for iterative search loops or third-party black-box solvers (like scipy.optimize) just to find $x$. Given any pressure state $(\lambda, \mu)$, the formula instantly yields the precise asset allocation vector that minimizes the current penalization landscape.

In [21]:
def get_x_star(lam, mu):
    """
    Computes the exact optimal portfolio allocation vector x for any 
    given penalization parameters via the derived first-order optimality condition.
    """
    return Sigma_inv @ (lam * r - mu * ones)

In [22]:
# Initialize dual variables (Shadow Prices) arbitrarily at a zero-pressure state
lam = 0.0 # Lagrange multiplier for Return Constraint (Inequality)
mu = 0.0 # Lagrange multiplier for Capital Constraint (Equality)

In [26]:
learning_rate = 0.005
epochs = 2001
x = get_x_star(lam, mu)

In [31]:
for epoch in range (epochs):

    x = get_x_star(lam, mu)

    grad_lam = R_target - (r @ x)
    grad_mu = ones @ x - 1

    lam += learning_rate * grad_lam
    mu += learning_rate * grad_mu

    # Bound Enforcement: Inequality multipliers cannot cross below zero
    if lam < 0:
        lam = 0.0
        
    # Log progress checkpoints
    if epoch % 500 == 0:
        portfolio_risk = 0.5 * (x @ Sigma @ x)
        g_x = R_target - (r @ x)
        h_x = (ones @ x) - 1
        L_val = portfolio_risk + lam * g_x + mu * h_x
        
        print(f"Epoch {epoch:4d} | portfolioRisk (P): {portfolio_risk:.6f}  | Lambda (g1): {lam:.4f} | Mu (h1): {mu:.4f} | Lower Bound (L): {L_val:.6f}")


Epoch    0 | portfolioRisk (P): 0.002114  | Lambda (g1): 0.0975 | Mu (h1): -0.0029 | Lower Bound (L): 0.002215
Epoch  500 | portfolioRisk (P): 0.002129  | Lambda (g1): 0.0999 | Mu (h1): -0.0028 | Lower Bound (L): 0.002217
Epoch 1000 | portfolioRisk (P): 0.002143  | Lambda (g1): 0.1019 | Mu (h1): -0.0028 | Lower Bound (L): 0.002219
Epoch 1500 | portfolioRisk (P): 0.002154  | Lambda (g1): 0.1037 | Mu (h1): -0.0028 | Lower Bound (L): 0.002220
Epoch 2000 | portfolioRisk (P): 0.002164  | Lambda (g1): 0.1051 | Mu (h1): -0.0028 | Lower Bound (L): 0.002221


In [29]:
print("\n--- OPTIMIZATION RESULTS SUMMARY ---")
print("Target Allocated Weights Vector (x*):")
for ticker, weight in zip(df.columns, x):
    print(f"  - {ticker}: {weight * 100:.2f}%")

print("\nConstraint Verification Metrics:")
print(f"  -> Total Budget Allocation (Target = 100%): {np.sum(x) * 100:.2f}%")
print(f"  -> Portfolio Expected Return (Target >= {R_target*100}%): {(r @ x) * 100:.2f}%")
print(f"  -> Minimized Portfolio Risk (Standard Deviation): {np.sqrt(x @ Sigma @ x):.6f}")


--- OPTIMIZATION RESULTS SUMMARY ---
Target Allocated Weights Vector (x*):
  - General Electric: -25.88%
  - Chesapeake Energy: -2.13%
  - AT&T: 54.67%
  - Cisco Systems: 38.20%
  - Campbell Soup: 35.13%

Constraint Verification Metrics:
  -> Total Budget Allocation (Target = 100%): 100.00%
  -> Portfolio Expected Return (Target >= 1.5%): 1.30%
  -> Minimized Portfolio Risk (Standard Deviation): 0.063670
